
# Symptom2Disease – Ensemble Diagnosis with MLP + Gradient Boosting and SHAP

This project proposes an ensemble disease diagnosis model that combines a **Multi-Layer Perceptron (MLP)** and 
**Gradient Boosting** to predict diseases from natural-language symptom descriptions using the **Symptom2Disease** 
dataset. Unlike prior single-model approaches, our ensemble integrates multiple optimizers and validation 
techniques to improve accuracy and generalization. To ensure transparency, we apply **SHAP (SHapley Additive 
exPlanations)** to identify which symptoms most influence each diagnosis. This combination of predictive power and 
interpretability aims to enhance trust and usability in AI-assisted medical diagnosis. 

## I. ---  
## II. ---  

## Introduction  

### What is the problem you will be investigating? 

Using a neural network, we propose to train a model that recommends/predicts a disease diagnosis using 
a patient’s description of their symptoms; we will be using the **Symptom2Disease** dataset to identify 
likely diseases from natural-language symptom inputs. Further, in addition to training a model to 
diagnose, we intend to use the model to learn what symptoms contributed the most to their diagnosis. 

### Why is this problem interesting or important? 

Accurate and early diagnosis is crucial for improving healthcare accessibility and patient outcomes. It is 
important because it can help with early diagnosis of the disease and can help people act sooner. Healthcare 
can be automated in a way that doesn’t (yet) involve having machines do the actual procedures. Instead, AI 
and machines can perform diagnosis based on patient feedback. 

### What are the key challenges or complexities involved in addressing this problem?  

Some of the key challenges are collecting a good quality data/dataset because medical records and 
information aren’t always accurate and complete and natural language is highly variable, and patients can 
describe the same symptom in many ways, which complicates interpretation. Additionally, symptoms may 
overlap across diseases, requiring the model to learn complex contextual relationships. 

## Related Work 

**Neural network-based disease prediction: Leveraging symptoms for accurate diagnosis of multiple 
diseases**  
https://www.researchgate.net/publication/391907900_Neural_network_based_disease_prediction_Leveraging_symptoms_for_accurate_diagnosis_of_multiple_diseases  

This paper talks about using a Simple MLP to predict the disease. Their dataset consists of ~130 
diseases and ~4900 data points. Using Keras and TensorFlow, they trained the NN. 

**Interpretable Machine Learning for Personalized Medical Recommendations: A LIME-Based Approach (PMC)**  

### How does your approach differ from or improve upon these existing methods?  

To begin with, the data is very different for both of us. Model wise, they used a simple MLP feedforward 
technique to recommend a diagnosis. However, we plan to use MLP and Gradient Boosting with 
hyperparameter tuning, optimizers, and various validation techniques. We also propose adding 
explainability using SHAP. 

### How can you effectively reference this work? 

`\citep{Barman2025}` – Machine Learning and Neural Networks have been at the forefront of Medical 
Diagnosis in [2]. We plan to extend this research in [1] by adding another layer of modeling using stacking. 

## III. The Methodology 

### What method or algorithm are you proposing? 

For the proposed solution to this problem, we will be using an ensemble algorithm with multi-layered 
perceptron (MLP) as the primary algorithm that takes vectorized symptom text as input and outputs a 
disease label that most accurately fits the scenario. We will pair this with Gradient Boosting. Using our data 
set, we will train the ensemble (MLP and GB) to take input from a user describing the symptoms they are 
experiencing and output a disease to diagnosis the user. 

### If there are existing implementations, how will you use or modify them? 

We plan to use an ensemble method as opposed to utilizing just one model. This will make it so that the 
ensemble can learn different patterns which will increase the accuracy of the predictions and will later 
apply hyperparameter tuning to find the optimal set of parameters.  

### How do you plan to improve or adapt existing methods to better solve the problem? 

We plan to use multiple models, optimizers, explainability, and validation techniques. This ensures better 
performance and accuracy. We will also use SHAP to explain which symptoms had the greatest influence 
on the diagnosis.    

## IV. Experimental Evaluation 

### What dataset(s) will you use? 

We are using the **Symptom2Disease** dataset, which has symptom descriptions mapped to disease labels. 
Before training, we will clean the data by removing duplicates, fixing similar and generalized symptom 
names, and classifying/splitting it into training and testing sets.  

### How will you evaluate your results? 

We will evaluate by utilizing a train/test split, which will allow us to test the model with symptom 
descriptions that the model did not see while training. We will use K-fold cross validation first and then 
evaluation metrics such as accuracy, macro F1-score, precision, recall, and top-k accuracy. At the end, we 
use LLMs (another form of NN) for secondary evaluation. This will help us see how good the model is at 
predicting diseases it hasn’t seen before. 

### Qualitatively, what kind of results do you expect?  

We plan to produce visuals like confusion matrices, ROC curves, and SHAP plots to show how well the 
model predicts and explain which symptoms influence each diagnosis. These results will help visualize 
both model accuracy and interpretability in an easy-to-understand way. 

### Quantitatively, what performance metrics or statistical tests will you use?  

We will evaluate performance using accuracy, precision, recall, F1 score, and Top-k accuracy to measure 
prediction quality. These will tell us how often the model makes the right predictions. The model’s results 
will be compared against simpler baselines like logistic regression to show clear improvements in accuracy 
and reliability.


In [ ]:

# ==============================
# Imports & Configuration
# ==============================
import warnings, os, json, random
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

# Keras API only (no explicit tensorflow / torch imports)
import keras
from keras import layers, callbacks, models

import shap  # for SHAP explainability

SEED      = 42
DATA_PATH = "Symptom2Disease.csv"  # adjust path if needed
TEXT_COL  = "text"
LABEL_COL = "label"

TEST_SIZE = 0.15
VAL_SIZE  = 0.15
TOP_K     = 3

random.seed(SEED)
np.random.seed(SEED)


In [ ]:

# ==============================
# Load & Clean Dataset
# ==============================
df = pd.read_csv(DATA_PATH).dropna(subset=[TEXT_COL, LABEL_COL]).copy()
df[TEXT_COL] = (
    df[TEXT_COL]
      .astype(str)
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
      .str.lower()
)
df = df.drop_duplicates(subset=[TEXT_COL, LABEL_COL]).reset_index(drop=True)

print(f"Rows: {len(df)}, Unique diseases: {df[LABEL_COL].nunique()}")
df.head()


In [ ]:

# ==============================
# Encode Labels & Split Data
# ==============================
le = LabelEncoder()
y = le.fit_transform(df[LABEL_COL].values)
X_text = df[TEXT_COL].values

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_text, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)
val_ratio = VAL_SIZE / (1 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=val_ratio, stratify=y_trainval, random_state=SEED
)

print(f"Splits → train: {len(X_train)}, val: {len(X_val)}, test: {len(X_test)}")

# ==============================
# TF-IDF Vectorization
# ==============================
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.9,
    max_features=15000,
    sublinear_tf=True
)

X_train_t = tfidf.fit_transform(X_train)
X_val_t   = tfidf.transform(X_val)
X_test_t  = tfidf.transform(X_test)

# Keras MLP will use dense arrays
X_train_d = X_train_t.toarray().astype("float32")
X_val_d   = X_val_t.toarray().astype("float32")
X_test_d  = X_test_t.toarray().astype("float32")

input_dim = X_train_d.shape[1]
n_classes = len(le.classes_)

print("Input dim:", input_dim, "| #classes:", n_classes)


In [ ]:

# ==============================
# Keras MLP Builder
# ==============================
def build_mlp(input_dim: int, n_classes: int, hidden1: int, hidden2: int, dropout: float, lr: float):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(hidden1, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(hidden2, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(n_classes, activation="softmax")
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)


## Hyperparameter Tuning (MLP + Gradient Boosting + Ensemble)

In [ ]:

# ==============================
# Simple Manual Hyperparameter Search
# ==============================
def topk_accuracy_from_probs(probs: np.ndarray, y_true: np.ndarray, k: int = 3) -> float:
    topk = np.argsort(-probs, axis=1)[:, :k]
    return float(np.mean([y_true[i] in topk[i] for i in range(len(y_true))]))

search_space = [
    {
        "name": "cfg1",
        "hidden1": 256,
        "hidden2": 64,
        "dropout": 0.2,
        "lr": 1e-3,
        "gb_n_estimators": 150,
        "gb_lr": 0.08,
        "gb_max_depth": 3
    },
    {
        "name": "cfg2",
        "hidden1": 256,
        "hidden2": 128,
        "dropout": 0.3,
        "lr": 5e-4,
        "gb_n_estimators": 200,
        "gb_lr": 0.05,
        "gb_max_depth": 3
    },
    {
        "name": "cfg3",
        "hidden1": 512,
        "hidden2": 128,
        "dropout": 0.3,
        "lr": 1e-3,
        "gb_n_estimators": 250,
        "gb_lr": 0.08,
        "gb_max_depth": 4
    }
]

best_score = -1.0
best_cfg   = None
best_mlp   = None
best_gb    = None
best_meta  = None

for cfg in search_space:
    print(f"\n=== Evaluating {cfg['name']} ===")
    # Build & train MLP
    mlp = build_mlp(input_dim, n_classes,
                    hidden1=cfg["hidden1"],
                    hidden2=cfg["hidden2"],
                    dropout=cfg["dropout"],
                    lr=cfg["lr"])
    mlp.fit(
        X_train_d, y_train,
        validation_data=(X_val_d, y_val),
        epochs=40,
        batch_size=128,
        verbose=0,
        callbacks=[early_stop]
    )
    # Gradient Boosting with this config
    gb = GradientBoostingClassifier(
        n_estimators=cfg["gb_n_estimators"],
        learning_rate=cfg["gb_lr"],
        max_depth=cfg["gb_max_depth"],
        subsample=0.9,
        random_state=SEED
    )
    gb.fit(X_train_d, y_train)

    # Validation ensemble via meta-learner
    p_mlp_val = mlp.predict(X_val_d, verbose=0)
    p_gb_val  = gb.predict_proba(X_val_d)
    Z_val     = np.hstack([p_mlp_val, p_gb_val])

    meta = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        multi_class="multinomial",
        solver="lbfgs",
        random_state=SEED
    )
    meta.fit(Z_val, y_val)

    # Evaluate on validation
    y_val_pred   = meta.predict(Z_val)
    p_val_meta   = meta.predict_proba(Z_val)
    macro_f1     = f1_score(y_val, y_val_pred, average="macro")
    topk_acc_val = topk_accuracy_from_probs(p_val_meta, y_val, k=TOP_K)

    print("Val macro F1:", macro_f1, "| Val Top-k:", topk_acc_val)

    # Use macro F1 as main selection metric
    if macro_f1 > best_score:
        best_score = macro_f1
        best_cfg   = cfg
        best_mlp   = mlp
        best_gb    = gb
        best_meta  = meta

print("\nBest config:", best_cfg)
print("Best Val macro F1:", best_score)


## Train Best Ensemble on Train+Val and Evaluate on Test

In [ ]:

# ==============================
# Retrain Best Configuration on Train+Val
# ==============================
X_trvall_d = np.vstack([X_train_d, X_val_d])
y_trvall   = np.concatenate([y_train, y_val])

# Rebuild MLP with best config and train on train+val
mlp_full = build_mlp(
    input_dim, n_classes,
    hidden1=best_cfg["hidden1"],
    hidden2=best_cfg["hidden2"],
    dropout=best_cfg["dropout"],
    lr=best_cfg["lr"]
)
mlp_full.fit(
    X_trvall_d, y_trvall,
    validation_split=0.1,
    epochs=40,
    batch_size=128,
    verbose=0,
    callbacks=[early_stop]
)

# Rebuild GB with best config and train on train+val
gb_full = GradientBoostingClassifier(
    n_estimators=best_cfg["gb_n_estimators"],
    learning_rate=best_cfg["gb_lr"],
    max_depth=best_cfg["gb_max_depth"],
    subsample=0.9,
    random_state=SEED
)
gb_full.fit(X_trvall_d, y_trvall)

# On test set, use ensemble with a new meta-learner trained on val Z (best_meta already fits val)
p_mlp_test = mlp_full.predict(X_test_d, verbose=0)
p_gb_test  = gb_full.predict_proba(X_test_d)
Z_test     = np.hstack([p_mlp_test, p_gb_test])

y_pred_test = best_meta.predict(Z_test)
p_meta_test = best_meta.predict_proba(Z_test)

test_metrics = {
    "accuracy":        float(accuracy_score(y_test, y_pred_test)),
    "macro_precision": float(precision_score(y_test, y_pred_test, average="macro", zero_division=0)),
    "macro_recall":    float(recall_score(y_test, y_pred_test, average="macro", zero_division=0)),
    "macro_f1":        float(f1_score(y_test, y_pred_test, average="macro")),
    "top_k_accuracy":  topk_accuracy_from_probs(p_meta_test, y_test, k=TOP_K)
}
print("Test metrics:", json.dumps(test_metrics, indent=2))


## Confusion Matrix (Top 20 Classes)

In [ ]:

from collections import Counter

cnt = Counter(y_test)
top_classes = [c for c, _ in cnt.most_common(20)]
mask = np.isin(y_test, top_classes)

cm = confusion_matrix(y_test[mask], y_pred_test[mask], labels=top_classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.inverse_transform(top_classes))

plt.figure(figsize=(10, 8))
disp.plot(values_format="d", xticks_rotation=90, cmap=None)
plt.title("Confusion Matrix (Top 20 Classes)")
plt.tight_layout()
plt.show()


## SHAP Explainability (GB Branch)

In [ ]:

# ==============================
# SHAP – Which Symptoms Influence Each Diagnosis?
# ==============================
# Use TreeExplainer on the best GB model over TF-IDF features
X_exp = X_test_t[:150].toarray()

try:
    explainer = shap.TreeExplainer(gb_full)
    shap_values = explainer.shap_values(X_exp)

    # Aggregate mean |SHAP| across classes if multiclass
    if isinstance(shap_values, list):
        mean_abs = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
    else:
        mean_abs = np.abs(shap_values).mean(axis=0)

    feature_names = tfidf.get_feature_names_out()
    top_idx = np.argsort(-mean_abs)[:20]
    top_terms = [(feature_names[i], float(mean_abs[i])) for i in top_idx]

    print("Top SHAP terms (global importance):")
    for term, score in top_terms:
        print(f"{term:25s} | {score:.6f}")

    # Bar plot (no explicit colors)
    plt.figure(figsize=(8, 6))
    y_pos = np.arange(len(top_idx))
    plt.barh(y_pos, mean_abs[top_idx])
    plt.yticks(y_pos, feature_names[top_idx])
    plt.gca().invert_yaxis()
    plt.title("Top Symptoms by SHAP Importance (GB branch)")
    plt.xlabel("Mean |SHAP value|")
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("SHAP computation skipped due to:", e)


## Save Artifacts & Inference Helper

In [ ]:

import joblib, pickle

# Save core components
joblib.dump(tfidf, "tfidf_vectorizer.joblib")
joblib.dump(le, "label_encoder.joblib")
joblib.dump(best_meta, "meta_logreg.joblib")
mlp_full.save("keras_mlp_best.keras")
with open("gb_full_best.pkl", "wb") as f:
    pickle.dump(gb_full, f)

print("Saved: tfidf_vectorizer.joblib, label_encoder.joblib, meta_logreg.joblib, keras_mlp_best.keras, gb_full_best.pkl")

def predict_symptoms(texts, top_k: int = 3):
    X = tfidf.transform(texts).toarray().astype("float32")
    p1 = mlp_full.predict(X, verbose=0)
    p2 = gb_full.predict_proba(X)
    Z  = np.hstack([p1, p2])
    p  = best_meta.predict_proba(Z)
    top = np.argsort(-p, axis=1)[:, :top_k]
    results = []
    for i, text in enumerate(texts):
        preds = [
            {"label": le.inverse_transform([j])[0], "prob": float(p[i, j])}
            for j in top[i]
        ]
        results.append({"input": text, "predictions": preds})
    return results

# Example
example_inputs = [
    "fever, dry cough, shortness of breath, fatigue",
    "abdominal pain with nausea and vomiting, low appetite"
]
print(json.dumps(predict_symptoms(example_inputs, top_k=TOP_K), indent=2))
